#

# Modelling embedding model on CT

In [1]:
from sentence_transformers import SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

from datasets import load_dataset, concatenate_datasets, Dataset

from tqdm.auto import tqdm
import regex as re
import os

import random

import torch.nn.functional as F
from torch.utils.data import DataLoader

from pprint import pprint
from torch.optim import AdamW

/tmp/ipykernel_14951/908894418.py:2: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
/tmp/ipykernel_14951/908894418.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
/tmp/ipykernel_14951/908894418.py:5: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


In [2]:
torch.cuda.empty_cache()

In [3]:
#!pip install sentence_transformers
import os
os.getcwd()

'/content'

### **Load data from hugginface and preprocess, train-test split, pre-process**

In [4]:
from datasets import load_dataset
import pandas as pd

In [5]:
repo_path = "vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final"
dataset = load_dataset(repo_path, split="train")

*Renaming columns*

In [6]:
len(dataset), type(dataset), dataset.column_names

(7866,
 datasets.arrow_dataset.Dataset,
 ['anchor_id',
  'nctId',
  'chunk_type',
  'block_no',
  'document_id',
  'anchor_type',
  'anchor',
  'positive',
  'chunk_char_length',
  'status'])

In [7]:
# Clean & Format Columns
if(re.search("data2$", repo_path)):#new data and old data final iteration
  dataset = dataset.rename_column("anchor_id", "id")
  dataset = dataset.rename_column("unique_chunk_identifier", "global_chunk_id")

  #remove 'document_id' and create map in iter2, ignoresame in iter1 for===>(3 anchor_5 chunk new data)
  dataset = dataset.remove_columns(['nctId', 'block_no', 'chunk_type', 'anchor_type', 'chunk_char_length', 'status'])

else:
  dataset = dataset.rename_column('anchor_id','id')
  dataset = dataset.rename_column('document_id','global_chunk_id')
  dataset = dataset.remove_columns(['nctId', 'block_no', 'chunk_type', 'anchor_type', 'chunk_char_length', 'status'])

### **Data preparation(seeded randomization & split--> construct query and corpus and relevant_docs)**

In [8]:
#We'll shuffle the dataset and do 90:10 train_test split with seed . This will aid easy loading and use for later metric calculations with multiple query_models.
dataset =dataset.shuffle(seed=42)

train_test_dataset = dataset.train_test_split(test_size=0.1, seed=42)

test_dataset = train_test_dataset["test"]
train_dataset = train_test_dataset["train"]

corpus_dataset = concatenate_datasets([train_dataset, test_dataset])# corpus =train+test combined

# Format: {query_id: question_text}
queries = dict(
    zip(test_dataset["id"], test_dataset["anchor"])
)

# Format: {corpus_id: text_chunk}
corpus = dict(
    zip(corpus_dataset["global_chunk_id"], corpus_dataset["positive"])
)

#map anchor and chunk identifiers(to establish anchor-positive pair reltn for test set)
relevant_docs ={}
for q_id, global_chunk_id in tqdm(zip(test_dataset["id"], test_dataset["global_chunk_id"])):
  relevant_docs[q_id] = [global_chunk_id]


0it [00:00, ?it/s]

Props check of dataset and its derivative datastructure

In [9]:
print(f"len_test_dataset:-{len(train_test_dataset['test'])}, len_train_dataset:- {len(train_test_dataset['train'])}")

df_test = train_test_dataset["test"].to_pandas()
print(f"test_unique_chunks_ids----{df_test["global_chunk_id"].nunique()} and test_unique_chunks-----{df_test["positive"].nunique()}")

len_test_dataset:-787, len_train_dataset:- 7079
test_unique_chunks_ids----683 and test_unique_chunks-----682


In [10]:
# Every relevant document must exist in corpus
assert all(
    doc_id in corpus
    for docs in relevant_docs.values()
    for doc_id in docs
)

# Every query must have a relevance entry
assert set(queries.keys()) == set(relevant_docs.keys())

print("Queries :", len(queries))
print("Corpus  :", len(corpus))
print("Relevant:", len(relevant_docs))

Queries : 787
Corpus  : 2000
Relevant: 787


### **Initialize both models , set doc_embed model to freeze; while query_encoder_model changes iteratively**

In [13]:
model_path_identifier_map={
    "nomic-ai/nomic-embed-text-v1.5":"nomic-embed_768",
    "NeuML/bioclinical-modernbert-base-embeddings":"bio-modern_bert_768",
    "sbiobert_base_cased_mli":"sbiobert_mli_768",
    "BAAI/bge-base-en-v1.5": "bge_768"
}

In [17]:
DOC_MODEL_ID = "vab46/nomic-embed-text-v1.5_Clinical-Trials_Matryoshka_final"
QUERY_MODEL_ID = "BAAI/bge-base-en-v1.5"# this will change for each baseline-finetune iteration

*Load the two encoders(with seeding)*

---



In [18]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# ---------------------------------------------------------
# 1. TRAINABLE QUERY ENCODER
# ---------------------------------------------------------
query_model = SentenceTransformer(QUERY_MODEL_ID, device=device)

# ---------------------------------------------------------
# 2. FROZEN DOCUMENT ENCODER
# ---------------------------------------------------------
doc_model = SentenceTransformer(DOC_MODEL_ID, device=device)

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

In [19]:
for param in doc_model.parameters():# by default its true
    param.requires_grad = False

print("Models initialized:", device)

Models initialized: cuda


### **Asymmetric retrieval evaluation**

The standard InformationRetrievalEvaluator(model=...) workflow is fundamentally designed around one model doing the encoding of both queries and corpus. It therefore isn't a clean fit for independently supplied query/document encoders.

In [20]:
def evaluate_retrieval(query_embeddings, corpus_embeddings, queries,
    corpus, relevant_docs, k_values=[1, 3, 5, 10], map_k=100):

    """
    Asymmetric retrieval evaluation.

    query_embeddings  : [num_queries, embedding_dim]
    corpus_embeddings : [num_documents, embedding_dim]

    queries           : {query_id: query_text}
    corpus            : {doc_id: document_text}
    relevant_docs     : {query_id: set/list of relevant_doc_ids}
    """

    # ---------------------------------------------------------
    # 1. Cosine similarity: query -> frozen document embeddings
    # ---------------------------------------------------------

    scores = cosine_similarity(query_embeddings, corpus_embeddings)

    corpus_ids = list(corpus.keys())
    query_ids = list(queries.keys())

    # ---------------------------------------------------------
    # 2. Ranking for every query(ranks of documents yeilded)
    # ---------------------------------------------------------

    rankings = {}

    for i, qid in enumerate(query_ids):

        ranked_indices = np.argsort(-scores[i])

        rankings[qid] = [
            corpus_ids[idx]
            for idx in ranked_indices
        ]

    # ---------------------------------------------------------
    # 3. Calculate metrics
    # ---------------------------------------------------------

    results = {}

    for k in k_values:
        accuracy_scores = []
        precision_scores = []
        recall_scores = []
        mrr_scores = []
        ndcg_scores = []

        for qid in query_ids:
            ranked_docs = rankings[qid][:k]
            relevant = set(relevant_docs[qid])

            # Relevant retrieved
            hits = [
                doc_id
                for doc_id in ranked_docs
                if doc_id in relevant
            ]

            num_hits = len(hits)

            # Accuracy / Hit Rate
            accuracy = 1.0 if num_hits > 0 else 0.0

            # Precision
            precision = num_hits / k

            # Recall
            recall = (
                num_hits / len(relevant)
                if len(relevant) > 0
                else 0.0
            )


            # MRR
            reciprocal_rank = 0.0

            for rank, doc_id in enumerate(ranked_docs, start=1):
                if doc_id in relevant:
                    reciprocal_rank = 1.0 / rank
                    break


            # NDCG
            dcg = 0.0

            for rank, doc_id in enumerate(ranked_docs, start=1):
                if doc_id in relevant:
                    dcg += 1.0 / np.log2(rank + 1)

            ideal_hits = min(k, len(relevant))

            idcg = sum(
                1.0 / np.log2(rank + 1)
                for rank in range(1, ideal_hits + 1)
            )

            ndcg = dcg / idcg if idcg > 0 else 0.0

            # Store
            accuracy_scores.append(accuracy)
            precision_scores.append(precision)
            recall_scores.append(recall)
            mrr_scores.append(reciprocal_rank)
            ndcg_scores.append(ndcg)

        # Aggregate across queries
        results[f"accuracy@{k}"] = np.mean(accuracy_scores)
        results[f"precision@{k}"] = np.mean(precision_scores)
        results[f"recall@{k}"] = np.mean(recall_scores)
        results[f"mrr@{k}"] = np.mean(mrr_scores)
        results[f"ndcg@{k}"] = np.mean(ndcg_scores)

    #MAP@100
    ap_scores = []

    for qid in query_ids:

        ranked_docs = rankings[qid][:map_k]
        relevant = set(relevant_docs[qid])

        if len(relevant) == 0:
            ap_scores.append(0.0)
            continue

        hits = 0
        precision_sum = 0.0

        for rank, doc_id in enumerate(ranked_docs, start=1):
            if doc_id in relevant:
                hits += 1
                precision_at_rank = hits / rank
                precision_sum += precision_at_rank

        denominator = min(len(relevant), map_k)
        ap = precision_sum / denominator
        ap_scores.append(ap)

    results[f"map@{map_k}"] = np.mean(ap_scores)

    return results


**Seed + initialize both encoders**

*Encode corpus and queries*

In [21]:
#for 1st run only
'''
corpus_ids = list(corpus.keys())
corpus_texts = [corpus[cid] for cid in corpus_ids]

with torch.no_grad():
    corpus_embeddings = doc_model.encode(
        corpus_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_tensor=True
    )

print(corpus_embeddings.shape)
'''

'\ncorpus_ids = list(corpus.keys())\ncorpus_texts = [corpus[cid] for cid in corpus_ids]\n\nwith torch.no_grad():\n    corpus_embeddings = doc_model.encode(\n        corpus_texts,\n        batch_size=32,\n        show_progress_bar=True,\n        normalize_embeddings=True,\n        convert_to_tensor=True\n    )\n\nprint(corpus_embeddings.shape)\n'

*Freeze these embeddings for re-usiblity(with diff query encoder models)*

In [22]:
'''
torch.save(
    {
        "corpus_ids": corpus_ids,
        "embeddings": corpus_embeddings.cpu()
    },
    "frozen_ct_nomic_corpus_embeddings.pt"
)#saves at content level
'''
#for subsequent_load, just upload froze_embeddings at /content level in disc
saved = torch.load("frozen_ct_nomic_corpus_embeddings.pt", map_location=device)

corpus_ids =saved['corpus_ids']
corpus_embeddings = saved['embeddings'].to(device)
corpus_texts = [corpus[cid] for cid in corpus_ids]
print(corpus_embeddings.shape)

# to check whether loaded variables from torch(corpus ids and embeddings) are equal to generated
'''
corpus_ids = saved["corpus_ids"] if(saved['corpus_ids'] == corpus_ids) else []
corpus_embeddings = saved["embeddings"].to(device) if torch.equal(saved["embeddings"].cpu(),
                                                                  corpus_embeddings.cpu()) else [].to(device)

print(len(corpus_ids))#--->o/p-2000
print(len(corpus_embeddings))#----->o/p-2000
'''

torch.Size([2000, 768])


' \ncorpus_ids = saved["corpus_ids"] if(saved[\'corpus_ids\'] == corpus_ids) else []\ncorpus_embeddings = saved["embeddings"].to(device) if torch.equal(saved["embeddings"].cpu(), \n                                                                  corpus_embeddings.cpu()) else [].to(device)\n\nprint(len(corpus_ids))#--->o/p-2000\nprint(len(corpus_embeddings))#----->o/p-2000\n'

In [28]:
#print(corpus_embeddings.shape)

torch.Size([2000, 768])


In [23]:
query_ids = list(queries.keys())
query_texts = [queries[qid] for qid in query_ids]

with torch.no_grad():
    query_embeddings = query_model.encode(
        query_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_tensor=True
    )

print(query_embeddings.shape)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

torch.Size([787, 768])


In [24]:
base_results = evaluate_retrieval(
    query_embeddings=query_embeddings.cpu().numpy(),
    corpus_embeddings=corpus_embeddings.cpu().numpy(),
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs
)

In [25]:
pprint(base_results)

{'accuracy@1': np.float64(0.0),
 'accuracy@10': np.float64(0.005082592121982211),
 'accuracy@3': np.float64(0.0),
 'accuracy@5': np.float64(0.0025412960609911056),
 'map@100': np.float64(0.001991522839541452),
 'mrr@1': np.float64(0.0),
 'mrr@10': np.float64(0.0007765071297472823),
 'mrr@3': np.float64(0.0),
 'mrr@5': np.float64(0.0005082592121982211),
 'ndcg@1': np.float64(0.0),
 'ndcg@10': np.float64(0.0017329103385653778),
 'ndcg@3': np.float64(0.0),
 'ndcg@5': np.float64(0.000983107515208492),
 'precision@1': np.float64(0.0),
 'precision@10': np.float64(0.0005082592121982211),
 'precision@3': np.float64(0.0),
 'precision@5': np.float64(0.0005082592121982211),
 'recall@1': np.float64(0.0),
 'recall@10': np.float64(0.005082592121982211),
 'recall@3': np.float64(0.0),
 'recall@5': np.float64(0.0025412960609911056)}


## **Training(query only)**

In [26]:
best_model_path =model_path_identifier_map[QUERY_MODEL_ID]
best_ndcg10 = -float("inf")

*Train data pre-process(not need repeat for diff base query encoders), hyperparameter tuning,relating sibling queries*

In [27]:
train_queries = train_dataset["anchor"]
train_docs = train_dataset["positive"]

# Frozen training-document embeddings(only 1st time)
'''
with torch.no_grad():
    train_doc_embeddings = doc_model.encode_document(
        train_docs,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_tensor=True
    )
'''

'\nwith torch.no_grad():\n    train_doc_embeddings = doc_model.encode_document(\n        train_docs,\n        batch_size=32,\n        show_progress_bar=True,\n        normalize_embeddings=True,\n        convert_to_tensor=True\n    )\n'

In [28]:
#print(train_doc_embeddings.shape)

In [29]:
#only 1st time

'''
torch.save(
    {
        "embeddings": train_doc_embeddings.cpu()},
    "frozen_ct_nomic_train_positive_embeddings.pt"
)#saves at content level
'''


saved_train = torch.load("frozen_ct_nomic_train_positive_embeddings.pt", map_location=device)
#print(torch.equal(saved_train["embeddings"].cpu(), train_doc_embeddings.cpu()))#--->true

train_doc_embeddings = saved_train['embeddings'].to(device)
print(train_doc_embeddings.shape)


torch.Size([7079, 768])


In [30]:
def query_forward(texts):
    features = query_model.preprocess(texts)
    #features = {k: v.to(device) for k, v in features.items()}

    features = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in features.items()
    }

    emb = query_model(features)["sentence_embedding"]
    return F.normalize(emb, p=2, dim=1)# normalize train embeddings-->q^​=∥q∥2​q​ & cosine(q,d)=q^​⋅d^

optimizer = AdamW(
    query_model.parameters(), lr=2e-5, weight_decay=0.01
)

batch_size, epochs, temperature = 32, 4, 0.05

# Group rows having the same positive document(to avoid false negative)-->anologous to relevant_doc
groups = {}
for i, doc in enumerate(train_docs):
    groups.setdefault(doc, []).append(i)
groups = list(groups.values())

In [31]:
for epoch in tqdm(range(epochs), desc='query_encoder_ft'):
    query_model.train()

    g = torch.Generator().manual_seed(SEED + epoch)
    order = torch.randperm(len(groups), generator=g).tolist()

    total_loss = 0.0

    for start in range(0, len(order), batch_size):
        selected = order[start:start + batch_size]

        idx = [
            groups[j][torch.randint(
                len(groups[j]), (1,), generator=g).item()]
            for j in selected
        ]

        q = query_forward([train_queries[i] for i in idx])
        d = train_doc_embeddings[idx]

        logits = (q @ d.T) / temperature
        labels = torch.arange(len(idx), device=device)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} |Train Loss: {total_loss:.4f}")

    # ---------- Validation: NDCG@10 ----------
    query_model.eval()


    with torch.no_grad():
        val_query_embeddings = query_forward(list(test_dataset['anchor']))


    val_results = evaluate_retrieval(
        query_embeddings= val_query_embeddings.cpu().numpy(),
        corpus_embeddings= corpus_embeddings.cpu().numpy(),
        queries= queries,
        corpus= corpus,
        relevant_docs= relevant_docs
    )

    ndcg10 = val_results["ndcg@10"]

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {total_loss:.4f} | "
        f"Val NDCG@10: {ndcg10:.4f}"
    )

    if ndcg10 > best_ndcg10:
        best_ndcg10 = ndcg10
        query_model.save_pretrained(best_model_path)

    print("validation_done")




query_encoder_ft:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 1/4 |Train Loss: 216.0558
Epoch 1/4 | Loss: 216.0558 | Val NDCG@10: 0.0319


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

validation_done
Epoch 2/4 |Train Loss: 159.3936
Epoch 2/4 | Loss: 159.3936 | Val NDCG@10: 0.1293


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

validation_done
Epoch 3/4 |Train Loss: 117.0476
Epoch 3/4 | Loss: 117.0476 | Val NDCG@10: 0.1912


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

validation_done
Epoch 4/4 |Train Loss: 99.0213
Epoch 4/4 | Loss: 99.0213 | Val NDCG@10: 0.2333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

validation_done


### **Final RAG-2 evaluation**

In [32]:
# SILO 4 — Fine-tuned Query Evaluation
# ============================================================

query_model_ft = SentenceTransformer(best_model_path, device="cuda" if torch.cuda.is_available() else "cpu")
query_model.eval()


with torch.no_grad():
    ft_query_embeddings = query_model.encode_query(
        list(test_dataset["anchor"]),
        batch_size=32,
        normalize_embeddings=True,
        convert_to_tensor=True,
        show_progress_bar=True
    )


ft_results = evaluate_retrieval(
    ft_query_embeddings.cpu().numpy(),
    corpus_embeddings.cpu().numpy(),
    queries,
    corpus,
    relevant_docs
)

#-------------------metrics on final_embedd_data(doc_embed_froze_model)i.e RAG1:-
with torch.no_grad():
    RAG1_query_embeddings = doc_model.encode_query(
        list(test_dataset["anchor"]),
        batch_size=32,
        normalize_embeddings=True,
        convert_to_tensor=True,
        show_progress_bar=True
    )

RAG1_embed_results = evaluate_retrieval(
    RAG1_query_embeddings.cpu().numpy(),
    corpus_embeddings.cpu().numpy(),
    queries,
    corpus,
    relevant_docs
)

print("\nRAG1_embed_results")
pprint(RAG1_embed_results)


print("\nBASELINE")
pprint(base_results)

print("\nQUERY-FT")
pprint(ft_results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]


RAG1_embed_results
{'accuracy@1': np.float64(0.5743329097839899),
 'accuracy@10': np.float64(0.7534942820838628),
 'accuracy@3': np.float64(0.6823379923761118),
 'accuracy@5': np.float64(0.7102922490470139),
 'map@100': np.float64(0.6376615910218189),
 'mrr@1': np.float64(0.5743329097839899),
 'mrr@10': np.float64(0.6339797503075775),
 'mrr@3': np.float64(0.6215586615840745),
 'mrr@5': np.float64(0.6279754341380771),
 'ndcg@1': np.float64(0.5743329097839899),
 'ndcg@10': np.float64(0.6629022339587075),
 'ndcg@3': np.float64(0.6371528296560194),
 'ndcg@5': np.float64(0.6486909116383128),
 'precision@1': np.float64(0.5743329097839899),
 'precision@10': np.float64(0.07534942820838628),
 'precision@3': np.float64(0.2274459974587039),
 'precision@5': np.float64(0.14205844980940277),
 'recall@1': np.float64(0.5743329097839899),
 'recall@10': np.float64(0.7534942820838628),
 'recall@3': np.float64(0.6823379923761118),
 'recall@5': np.float64(0.7102922490470139)}

BASELINE
{'accuracy@1': np.f

In [33]:
# Print header
print("Fine Tuned Model Evaluation Results")
print("-" * 85)
print(f"{'Metrics':15} {'RAG1_ft':>12} {'RAG2_baseline':>12} {'RAG2_ft':>12}")
print('-' * 85)

metric_key = RAG1_embed_results.keys()

for key in metric_key:
    print(f"{key:15} {RAG1_embed_results[key]:>12.4f} {base_results[key]:>12.4f} {ft_results[key]:>12.4f}")


df_RAG2 = pd.DataFrame([RAG1_embed_results, base_results, ft_results]).T
df_RAG2.columns = ['RAG1_ft', 'RAG2_baseline', 'RAG2_ft']
df_RAG2.to_excel(f"RAG2_encoder_{model_path_identifier_map[QUERY_MODEL_ID]}_froze_doc.xlsx", engine='openpyxl')

Fine Tuned Model Evaluation Results
-------------------------------------------------------------------------------------
Metrics              RAG1_ft RAG2_baseline      RAG2_ft
-------------------------------------------------------------------------------------
accuracy@1            0.5743       0.0000       0.1169
precision@1           0.5743       0.0000       0.1169
recall@1              0.5743       0.0000       0.1169
mrr@1                 0.5743       0.0000       0.1169
ndcg@1                0.5743       0.0000       0.1169
accuracy@3            0.6823       0.0000       0.2084
precision@3           0.2274       0.0000       0.0695
recall@3              0.6823       0.0000       0.2084
mrr@3                 0.6216       0.0000       0.1561
ndcg@3                0.6372       0.0000       0.1695
accuracy@5            0.7103       0.0025       0.2808
precision@5           0.1421       0.0005       0.0562
recall@5              0.7103       0.0025       0.2808
mrr@5                

In [44]:
#df_RAG2.head()